## Popcorn Hack 1: Check One SFI Part Record

**Prediction:** every field is present and "Auto Racing" is an accepted category, so all four checks are `True` and the record is complete.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Check SFI Part Fields
part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}
valid_categories = ["Auto Racing", "Drag Racing"]

has_product_name = part["product_name"] != ""
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"] != ""
has_effective_date = part["effective_date"] != ""

print("Has product name:", has_product_name)
print("Valid category:", valid_category)
print("Has spec number:", has_spec_number)
print("Has effective date:", has_effective_date)
print("Record complete:", has_product_name and valid_category and has_spec_number and has_effective_date)

**Actual output:**

```text
Has product name: True
Valid category: True
Has spec number: True
Has effective date: True
Record complete: True
```

**Changed-input test:** category `"Street Car"` and an empty effective date. **Prediction:** valid category `False`, has effective date `False`, record complete `False`.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 (changed input) - Street Car with no date
part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}
part["category"] = "Street Car"
part["effective_date"] = ""
valid_categories = ["Auto Racing", "Drag Racing"]

has_product_name = part["product_name"] != ""
valid_category = part["category"] in valid_categories
has_spec_number = part["spec_number"] != ""
has_effective_date = part["effective_date"] != ""

print("Has product name:", has_product_name)
print("Valid category:", valid_category)
print("Has spec number:", has_spec_number)
print("Has effective date:", has_effective_date)
print("Record complete:", has_product_name and valid_category and has_spec_number and has_effective_date)

**Actual output:**

```text
Has product name: True
Valid category: False
Has spec number: True
Has effective date: False
Record complete: False
```

## Popcorn Hack 2: Backend Create Rule

Create only when every required field is valid **and** the spec number is **not** already stored.

**Prediction:** spec `1.1` is already stored, so it is rejected. Changing it to `1.2` is accepted; a bad category is rejected even with a new spec.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - Backend Create Rule
part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}
valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]

def can_create(part):
    fields_ok = (
        part["product_name"] != ""
        and part["category"] in valid_categories
        and part["spec_number"] != ""
        and part["effective_date"] != ""
    )
    is_duplicate = part["spec_number"] in existing_spec_numbers
    return fields_ok and not is_duplicate

print("Duplicate case (1.1):", can_create(part))

new_part = dict(part, spec_number="1.2")
print("New case (1.2):", can_create(new_part))

bad_category = dict(new_part, category="Street Car")
print("Bad category case:", can_create(bad_category))

**Actual output:**

```text
Duplicate case (1.1): False
New case (1.2): True
Bad category case: False
```

Matched my prediction: one valid case (1.2) and two rejected cases (duplicate, bad category).

## Popcorn Hack 3: SFI Search Filter

A record matches if the query equals its product name **or** its exact spec number, **and** its category is accepted. The loop reads each record's own fields, so it still works if the data changes.

**Prediction:** query `"1.2"` matches only Multiple Disc Clutch Assemblies.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - SFI Search Filter

query = "1.2"

records = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"}
]

valid_categories = ["Auto Racing", "Drag Racing"]

for record in records:
    name_match = query == record["product_name"]
    spec_match = query == record["spec_number"]
    category_ok = record["category"] in valid_categories
    if (name_match or spec_match) and category_ok:
        print("Match:", record["product_name"])

**Actual output:**

```text
Match: Multiple Disc Clutch Assemblies
```

**Changed-input test:** query by full product name, with a record whose category is not accepted added to the data. **Prediction:** only the accepted Racing Flywheel Record prints.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 (changed input) - Search by product name
query = "Racing Flywheel Record"

records = [
    {"product_name": "Replacement Flywheels and Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.1"},
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Drag Racing", "spec_number": "1.2"},
    {"product_name": "Racing Flywheel Record", "category": "Auto Racing", "spec_number": "2.1"},
    {"product_name": "Racing Flywheel Record", "category": "Street Car", "spec_number": "9.9"}
]

valid_categories = ["Auto Racing", "Drag Racing"]

for record in records:
    name_match = query == record["product_name"]
    spec_match = query == record["spec_number"]
    category_ok = record["category"] in valid_categories
    if (name_match or spec_match) and category_ok:
        print("Match:", record["product_name"], "(spec " + record["spec_number"] + ")")

**Actual output:**

```text
Match: Racing Flywheel Record (spec 2.1)
```

## MCQ

1. `7 >= 7` is **A** True, because `>=` is true when the values are equal.
2. Valid only with a name AND a spec: **C** `and`, since both conditions must be true.
3. `duplicate_spec` is False, so `not duplicate_spec` is **A** True.
4. Accept Auto Racing or Drag Racing: **B** `category == 'Auto Racing' or category == 'Drag Racing'`. Each side of `or` must be a full comparison.

**MCQ 3.05: 4/4 | answers: A,C,A,B**

## Homework Hack: SFI Backend Validator

**Boolean logic:** each rule is a named Boolean. A record is valid only if *all* of these are true, so they are joined with `and`:
`has_name and valid_category and has_spec and has_date and not is_duplicate`. The duplicate rule is written positively (`is_duplicate`) and flipped with `not`, which matches the written rule "spec number NOT already in the backend". When a record is rejected, the `elif` chain reports the first rule it broke.

**Prediction:** record 1 ACCEPTED; record 2 REJECTED (invalid category); record 3 REJECTED (duplicate spec).

In [ ]:
# CODE_RUNNER: Homework Hack - SFI Backend Validator
existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

def validate_record(record):
    has_name = record["product_name"] != ""
    valid_category = record["category"] in valid_categories
    has_spec = record["spec_number"] != ""
    has_date = record["effective_date"] != ""
    is_duplicate = record["spec_number"] in existing_spec_numbers
    # every rule must pass, so the Booleans are joined with "and";
    # the duplicate rule is flipped with "not"
    is_valid = has_name and valid_category and has_spec and has_date and not is_duplicate

    if is_valid:
        return "ACCEPTED"
    elif not has_name:
        return "REJECTED (missing product name)"
    elif not valid_category:
        return "REJECTED (invalid category)"
    elif not has_spec:
        return "REJECTED (missing spec number)"
    elif not has_date:
        return "REJECTED (missing effective date)"
    else:
        return "REJECTED (duplicate spec number)"

test_records = [
    {"product_name": "Multiple Disc Clutch Assemblies", "category": "Auto Racing", "spec_number": "1.2", "effective_date": "Feb. 9, 2006"},
    {"product_name": "Replacement Flywheels", "category": "Street Car", "spec_number": "3.1", "effective_date": "Jan. 1, 2026"},
    {"product_name": "Existing Flywheel Record", "category": "Drag Racing", "spec_number": "1.1", "effective_date": "Jan. 1, 2026"}
]

for number, record in enumerate(test_records, start=1):
    print("Record", number, "-", record["spec_number"], "->", validate_record(record))

**Actual output:**

```text
Record 1 - 1.2 -> ACCEPTED
Record 2 - 3.1 -> REJECTED (invalid category)
Record 3 - 1.1 -> REJECTED (duplicate spec number)
```

Matched my prediction.

### Extra tests: empty fields

**Prediction:** empty name, empty spec and empty date are each REJECTED with that reason.

In [ ]:
# CODE_RUNNER: Homework Hack test - empty name, empty spec, empty date
existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

def validate_record(record):
    has_name = record["product_name"] != ""
    valid_category = record["category"] in valid_categories
    has_spec = record["spec_number"] != ""
    has_date = record["effective_date"] != ""
    is_duplicate = record["spec_number"] in existing_spec_numbers
    # every rule must pass, so the Booleans are joined with "and";
    # the duplicate rule is flipped with "not"
    is_valid = has_name and valid_category and has_spec and has_date and not is_duplicate

    if is_valid:
        return "ACCEPTED"
    elif not has_name:
        return "REJECTED (missing product name)"
    elif not valid_category:
        return "REJECTED (invalid category)"
    elif not has_spec:
        return "REJECTED (missing spec number)"
    elif not has_date:
        return "REJECTED (missing effective date)"
    else:
        return "REJECTED (duplicate spec number)"

test_records = [
    {"product_name": "", "category": "Auto Racing", "spec_number": "4.1", "effective_date": "Jan. 1, 2026"},
    {"product_name": "Clutch Cover", "category": "Drag Racing", "spec_number": "", "effective_date": "Jan. 1, 2026"},
    {"product_name": "Flexplate", "category": "Drag Racing", "spec_number": "5.1", "effective_date": ""}
]

for number, record in enumerate(test_records, start=1):
    print("Record", number, "->", validate_record(record))

**Actual output:**

```text
Record 1 -> REJECTED (missing product name)
Record 2 -> REJECTED (missing spec number)
Record 3 -> REJECTED (missing effective date)
```

**Checklist:** valid and invalid cases tested; duplicate spec tested (record 3); the `and` / `not` in the code match the written rules.

## Submission Notes

```text
Lesson: CSP 3.05 Boolean Expressions
Notebook: https://rydenb.github.io/portfolio/python/boolean-hw
MCQ 3.05: 4/4 | answers: A,C,A,B
Popcorn Hacks 1–3 completed during class and copied to notebook: yes
Homework tested: 1.2 Multiple Disc Clutch Assemblies -> ACCEPTED; 3.1 Street Car -> REJECTED (invalid category); 1.1 duplicate -> REJECTED (duplicate spec number); empty name / empty spec / empty date -> each REJECTED with that reason
```